In [1]:
import pandas as pd
import plotly.express as px
from bs4 import BeautifulSoup
from pathlib import Path
import re

# =========================
# 1) CHEMIN DES FICHIERS HTML
# =========================
dossier = Path(r"../data/IPC")   # غيّر هذا المسار
fichiers = sorted(dossier.glob("Analyse_*.html"))

# =========================
# 2) LECTURE AUTOMATIQUE
# =========================
resultats = []

for fichier in fichiers:
    mois = re.search(r"Analyse_(\d{4}_\d{2})", fichier.name).group(1)

    tables = pd.read_html(fichier)

    # Tableau 0 : évolution générale
    t0 = tables[0].copy()
    t0["Mois"] = mois

    for col in ["Evolution mensuelle", "Evolution annuelle"]:
        t0[col] = (
            t0[col].astype(str)
            .str.replace(",", ".", regex=False)
            .astype(float)
        )

    resultats.append(t0)

df = pd.concat(resultats, ignore_index=True)

# L'order de Date 
df["Date"] = pd.to_datetime(df["Mois"], format="%Y_%m")
df = df.sort_values("Date")

print("Aperçu des données :")
print(df)

# =========================
# 3) ANALYSE SYNTHÉTIQUE
# =========================
general = df[df["Secteur"].str.contains("General", case=False, na=False)]

print("\n===== Analyse IPC =====")
print("Moyenne évolution mensuelle :", round(general["Evolution mensuelle"].mean(), 2), "%")
print("Moyenne évolution annuelle :", round(general["Evolution annuelle"].mean(), 2), "%")

mois_max = general.loc[general["Evolution mensuelle"].idxmax()]
mois_min = general.loc[general["Evolution mensuelle"].idxmin()]

print("Plus forte hausse mensuelle :", mois_max["Mois"], "=", mois_max["Evolution mensuelle"], "%")
print("Plus forte baisse mensuelle :", mois_min["Mois"], "=", mois_min["Evolution mensuelle"], "%")

# =========================
# 4) GRAPHIQUE 1 : IPC Général
# =========================
fig1 = px.line(
    general,
    x="Date",
    y="Evolution mensuelle",
    markers=True,
    text="Evolution mensuelle",
    title="Evolution mensuelle de l’IPC général"
)

fig1.update_traces(textposition="top center")
fig1.update_layout(
    xaxis_title="Mois",
    yaxis_title="Evolution mensuelle (%)",
    template="plotly_white"
)

fig1.show()

# =========================
# 5) GRAPHIQUE 2 : Comparaison secteurs
# =========================
secteurs = ["General", "Alimentaire", "Non alimentaire"]

df_filtre = df[
    df["Secteur"].str.contains(
        "|".join(secteurs),
        case=False,
        na=False
    )
]

fig2 = px.line(
    df_filtre,
    x="Date",
    y="Evolution mensuelle",
    color="Secteur",
    markers=True,
    title="Comparaison mensuelle de l'IPC"
)

fig2.update_layout(
    xaxis_title="Mois",
    yaxis_title="Evolution mensuelle (%)",
    template="plotly_white"
)

fig2.show()
# =========================
# 6) GRAPHIQUE 3 : Evolution annuelle
# =========================
fig3 = px.line(
    df,
    x="Date",
    y="Evolution annuelle",
    color="Secteur",
    markers=True,
    title="Evolution annuelle de l’IPC par secteur"
)

fig3.update_layout(
    xaxis_title="Mois",
    yaxis_title="Evolution annuelle (%)",
    template="plotly_white"
)

fig3.show()
# =========================
# 7) TEXTE D’INTERPRÉTATION
# =========================
print("\n===== Interprétation courte =====")
print(
    "L’analyse montre une évolution variable de l’IPC entre les mois étudiés. "
    "Les produits alimentaires présentent généralement les variations les plus fortes, "
    "ce qui indique une sensibilité importante aux fluctuations des prix. "
    "Les produits non alimentaires restent plus stables. "
    "Ainsi, l’évolution globale de l’IPC est principalement influencée par le secteur alimentaire."
)

Aperçu des données :
                      Secteur  Evolution mensuelle  ...     Mois       Date
0                     General                  NaN  ...  2025_10 2025-10-01
1       Produits alimentaires                -16.0  ...  2025_10 2025-10-01
2   Produits non alimentaires                  1.0  ...  2025_10 2025-10-01
3                     General                 -9.0  ...  2025_11 2025-11-01
4       Produits alimentaires                -20.0  ...  2025_11 2025-11-01
5   Produits non alimentaires                  1.0  ...  2025_11 2025-11-01
7       Produits alimentaires                  2.0  ...  2025_12 2025-12-01
8   Produits non alimentaires                 -1.0  ...  2025_12 2025-12-01
6                     General                  1.0  ...  2025_12 2025-12-01
9                     General                  6.0  ...  2026_01 2026-01-01
10      Produits alimentaires                 16.0  ...  2026_01 2026-01-01
11  Produits non alimentaires                 -2.0  ...  2026_01 20


===== Interprétation courte =====
L’analyse montre une évolution variable de l’IPC entre les mois étudiés. Les produits alimentaires présentent généralement les variations les plus fortes, ce qui indique une sensibilité importante aux fluctuations des prix. Les produits non alimentaires restent plus stables. Ainsi, l’évolution globale de l’IPC est principalement influencée par le secteur alimentaire.
